# A Fresh look at Data Marshalling

Converting data from one format into an RDF graph goes through a number of stages. 

There needs to be some form of mapping that helps perform those steps - part of this was previously wrapped up in the "Serialization" ontology, but I think this needs careful review.

Ideally, there should be some simple configuration that enables such a mapping to take place, one that's relatively easy to construct.

In [1]:
import pandas as pd
import numpy as np
import rdflib

import networkx as nx
import gravis as gv
import uuid

In [2]:
pdm_df = pd.read_excel("Gemeral Payments Logical Model (DMCAR) v0.006.xlsx")

In [3]:
pdm_df.columns

Index(['Namespace', 'NamespaceLabel', 'NamespaceDescription', 'Domain',
       'DomainNamespace', 'DomainLabel', 'DomainDescription', 'ParentDomain',
       'DomainEvent', 'DomainEventLabel', 'DomainEventDescription',
       'DomainParticipant', 'DomainParticipantLabel',
       'DomainParticipantDescription', 'Model', 'ModelLabel',
       'ModelDescription', 'ModelType', 'Class', 'ClassLabel',
       'ClassDescription', 'Attribute', 'AttributeLabel',
       'AttributeDescription', 'Sequence', 'DataType', 'ValidValues',
       'ISO Mapping', 'Requires Fix/Rework after updates w/c 21/02/2024',
       'Nulls', 'IsPK', 'Relationship', '_uniquetest', 'RelationshipLabel',
       'RelationshipDescription', 'RelationshipType', 'FromNamespace',
       'FromClass', 'FromAttribute', 'FromCardinality', 'ToNamespace',
       'ToClass', 'ToAttribute', 'ToCardinality'],
      dtype='object')

In [4]:
ontology_g = rdflib.Graph()
ontology_g.parse ('../kgontologies/kgdmcar.owl', format='xml')
kgdmcar = rdflib.Namespace('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#')
ontology_g.bind('kgdmcar', kgdmcar)

In [5]:
q="""SELECT ?s ?n
WHERE {
    ?s <http://www.w3.org/2000/01/rdf-schema#subClassOf>+ <http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#NamedObject> .
    BIND (<http://www.w3.org/2000/01/rdf-schema#subClassOf> as ?p)
    BIND (<http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#NamedObject> as ?o)
    ?s <http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#MetaClassIsScopedWithin> ?n
}"""
ontology_results = {v for v in set(list(ontology_g.query(q)))}
naming_specifications = {k:v for k,v in ontology_results}
naming_specifications

{rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Model'): rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
 rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Attribute'): rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Class'),
 rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Class'): rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
 rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Domain'): rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
 rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'): rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
 rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Relationship'): rdfli

In [53]:
rdf_serialisation_config = {
    "GlobalVariables" : { "GlobalName" : "Value"},
    "NamedObjects" : { 
                       "NamespaceParent": {
                       "SerialisationLabel" : "NamespaceParent", 
                       "SerialisationParentLabel" : None,
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace")
                        }, 
                        "Namespace": {
                       "SerialisationLabel" : "Namespace", 
                       "SerialisationParentLabel" : "NamespaceParent",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace")
                        }, 
                       "DomainNamespace": {
                       "SerialisationLabel" : "DomainNamespace", 
                       "SerialisationParentLabel" : None,
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace")
                        }, 
                        "FromNamespace": {
                       "SerialisationLabel" : "FromNamespace", 
                       "SerialisationParentLabel" : None,
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace")
                        }, 
                       "ToNamespace": {
                       "SerialisationLabel" : "ToNamespace", 
                       "SerialisationParentLabel" : None,
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace")
                        }, 
                      "Domain" : {
                       "SerialisationLabel" : "Domain", 
                       "SerialisationParentLabel" : "DomainNamespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Domain")
                      },
                      "ParentDomain" : {
                       "SerialisationLabel" : "ParentDomain", 
                       "SerialisationParentLabel" : "DomainNamespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Domain")
                      },
                      "Model" : {
                       "SerialisationLabel" : "Model", 
                       "SerialisationParentLabel" : "Namespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Model")
                      },
                      "Class" : {
                       "SerialisationLabel" : "Class", 
                       "SerialisationParentLabel" : "Namespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Class")
                      },
                        "FromClass" : {
                       "SerialisationLabel" : "FromClass", 
                       "SerialisationParentLabel" : "FromNamespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Class")
                      },
                        "ToClass" : {
                       "SerialisationLabel" : "ToClass", 
                       "SerialisationParentLabel" : "ToNamespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Class")
                      },
                      
                      "Attribute" : {
                       "SerialisationLabel" : "Attribute", 
                       "SerialisationParentLabel" : "Class",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Attribute")
                      },
                      "ToAttribute" : {
                       "SerialisationLabel" : "ToAttribute", 
                       "SerialisationParentLabel" : "ToClass",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Attribute")
                      },
                      "FromAttribute" : {
                       "SerialisationLabel" : "FromAttribute", 
                       "SerialisationParentLabel" : "FromClass",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Attribute")
                      },
                      
                      "Relationship" : {
                       "SerialisationLabel" : "Relationship", 
                       "SerialisationParentLabel" : "Namespace",
                       "ModelClass" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Relationship")
                      },
                     },
    "Relationships" : { 
                        "AttributeIsContainedByClass": {
                            "DomainInstance" : "Attribute", 
                            "RangeInstance" : "Class", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#IsContainedBy")
                     },
                        "ClassIsContainedByDomain": {
                            "DomainInstance" : "Class", 
                            "RangeInstance" : "Domain", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#IsContainedBy")
                     },
                        "DomainIsContainedByDomain": {
                            "DomainInstance" : "Domain", 
                            "RangeInstance" : "ParentDomain", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#IsContainedBy")
                     },
                        "ClassIsContainedByModel": {
                            "DomainInstance" : "Class", 
                            "RangeInstance" : "Model", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#IsContainedBy")
                     },

                        "RelationshipIsContainedByModel": {
                            "DomainInstance" : "Relationship", 
                            "RangeInstance" : "Model", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#IsContainedBy")
                     },

                        "RelationshipToAttribute": {
                            "DomainInstance" : "Relationship", 
                            "RangeInstance" : "ToAttribute", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipToAttribute")
                     },

                        "RelationshipFromAttribute": {
                            "DomainInstance" : "Relationship", 
                            "RangeInstance" : "FromAttribute", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipFromAttribute")
                     },
                        "RelationshipToClass": {
                            "DomainInstance" : "Relationship", 
                            "RangeInstance" : "ToClass", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipToClass")
                     },

                        "RelationshipFromClass": {
                            "DomainInstance" : "Relationship", 
                            "RangeInstance" : "FromClass", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipFromClass")
                     },


                    },
    "Properties" : { 
                        "NamespaceName": {
                            "DomainInstance" : "Namespace", 
                            "RangeLiteral" : "Namespace", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Name")
                     },
                        "NamespaceLabel": {
                            "DomainInstance" : "Namespace", 
                            "RangeLiteral" : "NamespaceLabel", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Label")
                     },
                         "NamespaceDescription": {
                            "DomainInstance" : "Namespace", 
                            "RangeLiteral" : "NamespaceDescription", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Description")
                     },
                        "DomainName": {
                            "DomainInstance" : "Domain", 
                            "RangeLiteral" : "Domain", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Name")
                     },
                        "DomainLabel": {
                            "DomainInstance" : "Domain", 
                            "RangeLiteral" : "DomainLabel", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Label")
                     },
                        "DomainDescription": {
                            "DomainInstance" : "Domain", 
                            "RangeLiteral" : "DomainDescription", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Description")
                     },
                        "ModelName": {
                            "DomainInstance" : "Model", 
                            "RangeLiteral" : "Model", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Name")
                     },
                        "ModelLabel": {
                            "DomainInstance" : "Model", 
                            "RangeLiteral" : "ModelLabel", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Label")
                     },
                        "ModelDescription": {
                            "DomainInstance" : "Model", 
                            "RangeLiteral" : "ModelDescription", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Description")
                     },
                        "ModelType": {
                            "DomainInstance" : "Model", 
                            "RangeLiteral" : "ModelType", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#ModelHasType")
                     },
                        "ClassName": {
                            "DomainInstance" : "Class", 
                            "RangeLiteral" : "Class", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Name")
                     },
                        "ClassLabel": {
                            "DomainInstance" : "Class", 
                            "RangeLiteral" : "ClassLabel", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Label")
                     },
                        "ClassDescription": {
                            "DomainInstance" : "Class", 
                            "RangeLiteral" : "ClassDescription", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Description")
                     },

                        "AttributeName": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "Attribute", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Name")
                     },
                        "AttributeLabel": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "AttributeLabel", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Label")
                     },
                        "AttributeDescription": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "AttributeDescription", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Description")
                     },
                        "AttributeSequence": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "Sequence", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#AttributeHasSequenceNumber")
                     },
                        "AttributeDataType": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "DataType", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#AttributeHasDataType")
                     },
                        "AttributeNulls": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "Nulls", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#AttributeIsNullable")
                     },
                        "AttributeIsPK": {
                            "DomainInstance" : "Attribute", 
                            "RangeLiteral" : "IsPK", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#AttributeIsPK")
                     },

                        "RelationshipName": {
                            "DomainInstance" : "Relationship", 
                            "RangeLiteral" : "Relationship", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Name")
                     },
                        "RelationshipLabel": {
                            "DomainInstance" : "Relationship", 
                            "RangeLiteral" : "RelationshipLabel", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Label")
                     },
                        "RelationshipDescription": {
                            "DomainInstance" : "Relationship", 
                            "RangeLiteral" : "RelationshipDescription", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgnaming#Description")
                     },
                        "RelationshipType": {
                            "DomainInstance" : "Relationship", 
                            "RangeLiteral" : "RelationshipType", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipHasType")
                     },
                        "RelationshipFromCardinality": {
                            "DomainInstance" : "Relationship", 
                            "RangeLiteral" : "FromCardinality", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipFromCardinality")
                     },
                        "RelationshipToCardinality": {
                            "DomainInstance" : "Relationship", 
                            "RangeLiteral" : "ToCardinality", 
                            "Predicate" : rdflib.URIRef("http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#RelationshipToCardinality")
                     },
    },
}

In [42]:
def nan2None(value):
    if isinstance(value, type(np.nan)):
        if np.isnan(value):
            return None
    return value

In [43]:
# For each instance referenced in the configuration file, scan over the available data and extract the key
# label, parentlabel and class definitions expected in the data-file.
class_sets_d={}
lineage_graph=nx.MultiDiGraph()
unique_nodes=set()
for instance_name, instance_parameters in rdf_serialisation_config['NamedObjects'].items():
    if instance_parameters['ModelClass'] in naming_specifications.keys():
        print("##########################")
        print("##                      ##")
        print(f"##    {instance_name}")
        print("##                      ##")
        print("##########################")
        if instance_parameters['ModelClass'] not in class_sets_d.keys():
            class_sets_d[instance_parameters['ModelClass']]=set()
        for i, row in pdm_df.iterrows():
            #print(i, row)
            if nan2None(row.get(instance_parameters["SerialisationLabel"])) not in ('',None, 'None', np.nan):
                #print("`" + nan2None(row.get(instance_parameters["SerialisationLabel"])) + "`")
                node_values = tuple([i, 
                    instance_parameters['ModelClass'],
                    row.get(instance_parameters["SerialisationLabel"]), 
                    naming_specifications[instance_parameters['ModelClass']],
                    nan2None(row.get(instance_parameters["SerialisationParentLabel"]))])
                class_sets_d[instance_parameters['ModelClass']].add(node_values)
                node_data = frozenset({"instance_name":instance_name, 
                                       "node_class":node_values[1], 
                                       "label":node_values[2], 
                                       "parent_class":node_values[3], 
                                       "parent_label":node_values[4], 
                                       "parent_instance":instance_parameters["SerialisationParentLabel"]}.items())

                if node_data not in unique_nodes:
                    lineage_graph.add_node(uuid.uuid4().hex,**{k:v for k,v in node_data})
                    unique_nodes.add(node_data)
                
            #else:
                #print(i, row.get(instance_parameters["SerialisationLabel"]))
                #print("!", row.get(instance_parameters["SerialisationLabel"]), type(row.get(instance_parameters["SerialisationLabel"])))
    else:
        print (instance_parameters['ModelClass'])


##########################
##                      ##
##    NamespaceParent
##                      ##
##########################
##########################
##                      ##
##    Namespace
##                      ##
##########################
##########################
##                      ##
##    DomainNamespace
##                      ##
##########################
##########################
##                      ##
##    FromNamespace
##                      ##
##########################
##########################
##                      ##
##    ToNamespace
##                      ##
##########################
##########################
##                      ##
##    Domain
##                      ##
##########################
##########################
##                      ##
##    ParentDomain
##                      ##
##########################
##########################
##                      ##
##    Model
##                      ##
#####################

In [44]:
d=7
node=list(lineage_graph.nodes())[d]
lineage_graph.nodes(data=True)[node]

{'node_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Domain'),
 'parent_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
 'parent_label': 'Domains',
 'parent_instance': 'DomainNamespace',
 'instance_name': 'Domain',
 'label': 'MessagingDomain'}

In [45]:
[(n,d) for n,d in lineage_graph.nodes(data=True) if d['label']=='Domains']

[('5edd376cfb5349de8a679c7787dcb73c',
  {'parent_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
   'node_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
   'instance_name': 'Namespace',
   'label': 'Domains',
   'parent_instance': 'NamespaceParent',
   'parent_label': None}),
 ('445463bbfb794d4c9c60ccda3c322dc1',
  {'parent_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
   'node_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
   'label': 'Domains',
   'parent_instance': None,
   'instance_name': 'DomainNamespace',
   'parent_label': None})]

In [46]:
def get_nodes_from_list(graph, nlist):
    returnlist = []
    nodedata = graph.nodes(data=True)
    for n in nlist:
        data=nodedata[n]
        returnlist.append((n,data))
    return returnlist


def get_fully_qualified_name(graph, node):
    acc=[node]
    tnode = node
    finished=False
    while not finished:
        p = list(graph.predecessors(node))
        if len(p)==1:
            acc.append(p[0])
            node = p[0]
        elif len(p)==0:
            return ".".join([d['label'] for n,d in get_nodes_from_list(graph, acc)][::-1])


def search_lineage(graph, **kwargs):
    # Retrieve a {node:data} dictionary from graph where
    # kwarg values are all exact matches
    node_content = {k : d for k,d in graph.nodes(data=True) if all([v==d[k] for k,v in kwargs.items()])}
    return node_content
    

In [47]:
search_lineage(lineage_graph, instance_name="Namespace", label="Domains")

{'5edd376cfb5349de8a679c7787dcb73c': {'parent_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
  'node_class': rdflib.term.URIRef('http://www.semanticweb.org/tomk/ontologies/2025/5/kgdmcar#Namespace'),
  'instance_name': 'Namespace',
  'label': 'Domains',
  'parent_instance': 'NamespaceParent',
  'parent_label': None}}

In [48]:
# Build the name-lineage graph to establish fully-qualified-names based on parental names etc

for node, data in lineage_graph.nodes(data=True):
    
    #possible_parents = [n for n,d in lineage_graph.nodes(data=True) if (data['parent_instance']==d['instance_name']) and (data['parent_label']==d['label']) and (n!=node)]

    possible_parents = [n for n,d in search_lineage(lineage_graph, instance_name=data['parent_instance'], label=data['parent_label']).items() if n != node]
    if len (possible_parents)==1:
        lineage_graph.add_edge(possible_parents[0], node)
    elif len(possible_parents)==0:
        print(f"{node}: {data['instance_name']}({data['label']}) is a root-node")
    else:
        print("Danger!", f"{node} can't find unique interpretation!")
        print(data)
        print()
        print(possible_parents)
    #print(data['instance_name'], node, possible_parents, data['label'], data['parent_label'])
    

5edd376cfb5349de8a679c7787dcb73c: Namespace(Domains) is a root-node
1848851aaea343afa814f11329d7f4eb: Namespace(PaymentsLogical) is a root-node
445463bbfb794d4c9c60ccda3c322dc1: DomainNamespace(Domains) is a root-node
b04f002e4b264f0b8e2826d49f355770: FromNamespace(PaymentsLogical) is a root-node
0940080a0b6542deae69c2a344e30214: ToNamespace(PaymentsLogical) is a root-node


In [49]:
## Update the lineage_graph to apply the calculated fully_qualified_names

nx.set_node_attributes(lineage_graph, {n:get_fully_qualified_name(lineage_graph, n) for n in lineage_graph.nodes()}, name="fully_qualified_name")


In [50]:
#sorted([(n,d) for n,d in lineage_graph.nodes(data='fully_qualified_name')], key = lambda x : x[1])

In [51]:
## N.B. Raw Results are repeated for each row, needs a set-based method to reduce to discrete set

unique_relation_set=set()

for instance_name, instance_parameters in rdf_serialisation_config['Relationships'].items():
    print(instance_name)
    for i, row in pdm_df.iterrows():
        if instance_parameters['DomainInstance'] in row.keys() and instance_parameters['RangeInstance'] in row.keys():
            #print(row[instance_parameters['DomainInstance']], "-->", row[instance_parameters['RangeInstance']])
            domain_options=search_lineage(lineage_graph, instance_name=instance_parameters['DomainInstance'], label=row[instance_parameters['DomainInstance']])
            range_options=search_lineage(lineage_graph, instance_name=instance_parameters['RangeInstance'], label=row[instance_parameters['RangeInstance']])
            if len(domain_options)==1:
                domain_values = list(domain_options.items())[0]
            if len(range_options)==1:
                range_values = list(range_options.items())[0]
            if len(domain_options)==1 and len(range_options)==1:
                #print (domain_values[1]['fully_qualified_name'], f"--{instance_name}-->", range_values[1]['fully_qualified_name'])
                rel = tuple([domain_values[1]['fully_qualified_name'], instance_name, range_values[1]['fully_qualified_name']])
                if rel not in unique_relation_set:
                    unique_relation_set.add(rel)
                    


unique_relation_set

AttributeIsContainedByClass
ClassIsContainedByDomain
DomainIsContainedByDomain
ClassIsContainedByModel
RelationshipIsContainedByModel
RelationshipToAttribute
RelationshipFromAttribute
RelationshipToClass
RelationshipFromClass


{('Domains.AccountsDomain',
  'DomainIsContainedByDomain',
  'Domains.PaymentsDomain'),
 ('Domains.InterfaceDomain',
  'DomainIsContainedByDomain',
  'Domains.PaymentsDomain'),
 ('Domains.InvestigationDomain',
  'DomainIsContainedByDomain',
  'Domains.PaymentsDomain'),
 ('Domains.MessagingDomain',
  'DomainIsContainedByDomain',
  'Domains.PaymentsDomain'),
 ('Domains.PartyDomain',
  'DomainIsContainedByDomain',
  'Domains.PaymentsDomain'),
 ('Domains.Payment', 'DomainIsContainedByDomain', 'Domains.PaymentsDomain'),
 ('Domains.PaymentsDomain', 'DomainIsContainedByDomain', 'Domains.Root'),
 ('Domains.ProcessingDomain',
  'DomainIsContainedByDomain',
  'Domains.PaymentsDomain'),
 ('PaymentsLogical.Booking',
  'ClassIsContainedByDomain',
  'Domains.AccountsDomain'),
 ('PaymentsLogical.Booking.BookingDate',
  'AttributeIsContainedByClass',
  'PaymentsLogical.Booking'),
 ('PaymentsLogical.Booking.BookingReference',
  'AttributeIsContainedByClass',
  'PaymentsLogical.Booking'),
 ('PaymentsLog

In [54]:
## Repeat the process for the properties (n.b that assignment of fully_qualified_name properties will be brute-forced, rather than driven from config)

unique_property_set=set()

for instance_name, instance_parameters in rdf_serialisation_config['Properties'].items():
    print(instance_name)
    for i, row in pdm_df.iterrows():
        if instance_parameters['DomainInstance'] in row.keys() and str(row[instance_parameters['RangeLiteral']])!="":
            #print(row[instance_parameters['DomainInstance']], "-->", row[instance_parameters['RangeInstance']])
            domain_options=search_lineage(lineage_graph, instance_name=instance_parameters['DomainInstance'], label=row[instance_parameters['DomainInstance']])

            if len(domain_options)==1:
                domain_values = list(domain_options.items())[0]

            if len(domain_options)==1 :
                #print (domain_values[1]['fully_qualified_name'], f"--{instance_name}-->", range_values[1]['fully_qualified_name'])
                rel = tuple([domain_values[1]['fully_qualified_name'], instance_name, row[instance_parameters['RangeLiteral']]])
                if rel not in unique_property_set:
                    unique_property_set.add(rel)
                    


unique_property_set

NamespaceName
NamespaceLabel
NamespaceDescription
DomainName
DomainLabel
DomainDescription
ModelName
ModelLabel
ModelDescription
ModelType
ClassName
ClassLabel
ClassDescription
AttributeName
AttributeLabel
AttributeDescription
AttributeSequence
AttributeDataType
AttributeNulls
AttributeIsPK
RelationshipName
RelationshipLabel
RelationshipDescription
RelationshipType
RelationshipFromCardinality
RelationshipToCardinality


{('PaymentsLogical.PaymentInboundInterface', 'RelationshipDescription', nan),
 ('PaymentsLogical.Interface', 'ClassName', 'Interface'),
 ('PaymentsLogical.PostalAddress.UnitNumber', 'AttributeLabel', 'UnitNumber'),
 ('PaymentsLogical.InstructionMessageMessage',
  'RelationshipLabel',
  'PaymentMessage'),
 ('PaymentsLogical.PostalAddress.UnstructuredAddressLine6',
  'AttributeDataType',
  'String'),
 ('PaymentsLogical.PaymentInstruction.InstructionIdentifier',
  'AttributeNulls',
  'Yes'),
 ('PaymentsLogical.PartyIdentification.IdentityIssuer',
  'AttributeNulls',
  'Yes'),
 ('PaymentsLogical.Payment.OutboundInterfaceID', 'AttributeNulls', 'No'),
 ('PaymentsLogical.PartyIdentification.CountryOfBirth',
  'AttributeNulls',
  'Yes'),
 ('PaymentsLogical.PaymentInstruction.SettlementPriority',
  'AttributeDescription',
  'Indicator of the urgency or order of importance that the instructing party would like the instructed party to apply to the processing of the settlement instruction.\nValues

In [17]:
assert False

AssertionError: 

In [ ]:
[(n,d) for n,d in lineage_graph.nodes(data=True) if d['label']=='Payment']

In [ ]:
[(n,d) for n,d in lineage_graph.nodes(data=True) if d['instance_name']=='Namespace']

In [ ]:
gv.d3(lineage_graph)

In [ ]:
# For each instance referenced in the configuration file, scan over the available data and extract the key
# label, parentlabel and class definitions expected in the data-file.
class_sets_d={}
for instance_name, instance_parameters in rdf_serialisation_config['NamedObjects'].items():
    if instance_parameters['ModelClass'] in naming_specifications.keys():
        print("##########################")
        print("##                      ##")
        print(f"##    {instance_name}")
        print("##                      ##")
        print("##########################")
        if instance_parameters['ModelClass'] not in class_sets_d.keys():
            class_sets_d[instance_parameters['ModelClass']]=set()
        for i, row in pdm_df.iterrows():
            #print(i, row)
            if nan2None(row.get(instance_parameters["SerialisationLabel"])) not in ('',None, 'None', np.nan):
                #print("`" + nan2None(row.get(instance_parameters["SerialisationLabel"])) + "`")
                name_values = ".".join([v for v in [nan2None(row.get(instance_parameters["SerialisationLabel"])), nan2None(row.get(instance_parameters["SerialisationParentLabel"]))] if v is not None][::-1])
                class_sets_d[instance_parameters['ModelClass']].add(name_values)
            #else:
                #print("!", row.get(instance_parameters["SerialisationLabel"]), type(row.get(instance_parameters["SerialisationLabel"])))
    else:
        print (instance_parameters['ModelClass'])


In [ ]:
def disjoint_items(dict_of_sets):
    set_membership_d=dict()
    for i in set([i for s in dict_of_sets.values() for i in s]):
        set_membership_d[i]=set()
        for k,v in dict_of_sets.items():
            
            if i in v:
                set_membership_d[i].add(k)
    return set_membership_d

results = disjoint_items(class_sets_d)    
for k,v in results.items():
    if len(v)>1:
        print(k,v)
        

In [ ]:
# Find objects that share names, despite belonging to different type/classes
# We need to disambiguate this collection.

set_membership_d=dict()
for i in set([i for s in class_sets_d.values() for i in s]):
    set_membership_d[i]=set()
    for k,v in class_sets_d.items():
        
        if i in v:
            set_membership_d[i].add(k)

for k,v in set_membership_d.items():
    if len(v)>1:
        print(k,v)
            

In [ ]:
#### Convert SHACL file into Graph 
shacl_g = rdflib.Graph()
shacl_g.parse ('../kgontologies/kgnameShape.ttl', format='ttl')

q="""SELECT ?s ?p ?o
WHERE {
    ?s ?p ?o.
}"""
results = list(shacl_g.query(q))
